# Imports

In [1]:
from controller import Controller
from eom import EOM
import numpy as np
from matplotlib import pyplot as plt
from scipy.linalg import solve_continuous_are

# Set up EOM

In [2]:
eom = EOM()

# Params
mass = 1.625  # kg
inertia = np.diag([0.02, 0.02, 0.03])  # kg*m^2
leg_length = 0.15  # m
k_f_val = 62.8 # Slope of thrust/motor torque curve (linear)
k_yaw_val = 1.0 # Yaw torque constant, default 1.0

eom.set_parameters(mass, inertia, leg_length, k_f_val, k_yaw_val)
eom.setup()

# Define LQR function

In [3]:
def lqr(A: np.ndarray, B: np.ndarray, Q: np.ndarray, R: np.ndarray) -> np.ndarray:
    """Compute the LQR gain matrix K.

    Args:
        A (np.ndarray): State matrix.
        B (np.ndarray): Input matrix.
        Q (np.ndarray): State cost matrix.
        R (np.ndarray): Input cost matrix.

    Returns:
        np.ndarray: LQR gain matrix K.
    """
    P = solve_continuous_are(A, B, Q, R)
    K = np.linalg.inv(R) @ B.T @ P
    return K

# Create controller object

In [4]:
# Params
dt = 0.05  # time step, s
t_max = 120.0 # total simulation time, s
max_motor_torque = 0.0851209 # N*m
x0 = np.zeros(12)  # initial state

controller = Controller(eom, dt, t_max, max_motor_torque, x0)

# Define gain matrix K

In [5]:
Q = np.diag([10, 10, 10, 100, 100, 100, 1, 1, 1, 1, 1, 1])
R = np.diag([0.1, 0.1, 0.1, 0.1])

K = lqr(eom.A_num, eom.B_num, Q, R)

# Example control loop

TODO: Change x_des to match the goals of this drone

In [6]:
t = 0.0
x = x0
x_des = ...
while t < t_max:
    controller.step(t, x, x_des)
    t += dt
    x = controller.states[-1]

TypeError: bad operand type for unary -: 'NoneType'

# Plots and Analysis

In [8]:
controller.plot()